# DAILY SIMULATION

In [1]:
# import libraries and formulas
import time
import duckdb
import pandas as pd
from datetime import date, timedelta

from cpi_hard_method import (
    get_lastest_data,
    get_old_columns,
    parse_old_date,
    append_data,
    truncate_data,
    incremental_data,
    initialize_db,
    db_file,
    file,
)

Loading Excel file (once)...
Loaded 949 rows, 328 vintage columns.


In [ ]:
raw = pd.read_excel(
    file,
    na_values=["#N/A", "NA", "N/A", ""],
    engine="openpyxl",
)
raw.columns = raw.columns.str.strip()

# change first column name to date
raw = raw.rename(columns={raw.columns[0]: "DATE"})

old_cols = get_old_columns(raw)

# print total
print(f"Total old columns : {len(old_cols)}")
print(
    f"Earliest old      : {old_cols[0]}  ({parse_old_date(old_cols[0]).strftime('%Y-%m')})"
)
print(
    f"Latest old        : {old_cols[-1]}  ({parse_old_date(old_cols[-1]).strftime('%Y-%m')})"
)
print(f"Total observation rows: {len(raw)}")
print()
print("Preview (first 5 rows, first 4 columns):")
display(raw.iloc[:5, :4])


Total old columns : 328
Earliest old      : PCPI98M11  (1998-11)
Latest old        : PCPI26M2  (2026-02)
Total observation rows: 949

Preview (first 5 rows, first 4 columns):


,DATE,PCPI98M11,PCPI98M12,PCPI99M1
0,1947:01,NaN,NaN,NaN
1,1947:02,NaN,NaN,NaN
2,1947:03,NaN,NaN,NaN
3,1947:04,NaN,NaN,NaN
4,1947:05,NaN,NaN,NaN


# Testing get_lastest_data

In [3]:
for download_date in ["2004-01-15", "2005-02-15", "2025-02-15"]:
    df = get_lastest_data(download_date)
    print(
        f"download_date={download_date} -> {len(df)}, rows, last date: {df['DATE'].max()}"
    )

download_date=2004-01-15 -> 684, rows, last date: 2003:12
download_date=2005-02-15 -> 697, rows, last date: 2005:01
download_date=2025-02-15 -> 937, rows, last date: 2025:01


## Start from empty Table

In [4]:
with duckdb.connect(db_file) as con:
    for table in ["cpi_append", "cpi_trunc", "cpi_inc"]:
        con.execute(f"DROP TABLE IF EXISTS {table}")
        print(f" dropped{table}")
initialize_db()
print("Tables recreated empty")

 droppedcpi_append
 droppedcpi_trunc
 droppedcpi_inc
Tables recreated empty


## Daily Simulation Loop

In [5]:
start = date(2004, 1, 1)
end = date(2005, 12, 31)
total_days = (end - start).days + 1

print(f"Simulating {total_days} days ({start} -> {end}) ...")

timings = {"append": [], "trunc": [], "inc": []}
dates = []

with duckdb.connect(db_file) as con:
    for i in range(total_days):
        pull_date = (start + timedelta(days=i)).strftime("%Y-%m-%d")
        dates.append(pull_date)

        t0 = time.perf_counter()
        con.execute("BEGIN TRANSACTION")
        append_data(con, pull_date)
        con.execute("COMMIT")
        timings["append"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        truncate_data(con, pull_date)
        timings["trunc"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        con.execute("BEGIN TRANSACTION")
        incremental_data(con, pull_date)
        con.execute("COMMIT")
        timings["inc"].append(time.perf_counter() - t0)

        if i % 100 == 0:
            print(f"  {pull_date} ...")

print("Done.")


Simulating 731 days (2004-01-01 -> 2005-12-31) ...
  2004-01-01 ...
  2004-04-10 ...
  2004-07-19 ...
  2004-10-27 ...
  2005-02-04 ...
  2005-05-15 ...
  2005-08-23 ...
  2005-12-01 ...
Done.


## Speed Test

In [6]:
timing_df = pd.DataFrame(timings, index=dates)

print("Mean time per run (ms):")
print((timing_df.mean() * 1000).round(2).to_string())
print()
print("Total time (seconds):")
print(timing_df.sum().round(3).to_string())

Mean time per run (ms):
append      2.43
trunc     179.96
inc         5.04

Total time (seconds):
append      1.778
trunc     131.550
inc         3.684


# Discusion
append method has the fastest time while truncate method had the slowest time. 